In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## الخطوة 1: تعريف نماذج Pydantic للمخرجات المهيكلة

تحدد هذه النماذج **المخطط** الذي سيعيده الوكلاء. يضمن استخدام `response_format` مع Pydantic ما يلي:
- ✅ استخراج بيانات آمن بالنوع
- ✅ التحقق التلقائي
- ✅ عدم وجود أخطاء تحليل من ردود النص الحر
- ✅ توجيه شرطي سهل بناءً على الحقول


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## الخطوة 2: إنشاء أداة حجز الفندق

هذه الأداة هي ما سيتصل به **availability_agent** للتحقق مما إذا كانت الغرف متاحة. نستخدم المزين `@ai_function` لـ:
- تحويل دالة بايثون إلى أداة يمكن للذكاء الاصطناعي استدعاؤها
- إنشاء مخطط JSON تلقائيًا لـ LLM
- معالجة التحقق من صحة المعلمات
- تمكين الاستدعاء التلقائي بواسطة الوكلاء

لهذا العرض التوضيحي:
- **ستوكهولم، سياتل، طوكيو، لندن، أمستردام** → لديهم غرف ✅
- **جميع المدن الأخرى** → لا توجد غرف ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## الخطوة 3: تعريف دوال الشرط للتوجيه

تقوم هذه الدوال بفحص استجابة الوكيل وتحديد المسار الذي يجب اتباعه في سير العمل.

**النمط الرئيسي:**
1. تحقق مما إذا كانت الرسالة هي `AgentExecutorResponse`
2. تحليل الناتج الهيكلي (نموذج Pydantic)
3. إرجاع `True` أو `False` للتحكم في التوجيه

سيقوم سير العمل بتقييم هذه الشروط على **الحواف** لتحديد أي منفذ تنفيذ يجب استدعاؤه بعد ذلك.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## الخطوة 4: إنشاء منفذ عرض مخصص

المنفذون هم مكونات سير العمل التي تقوم بالتحويلات أو التأثيرات الجانبية. نستخدم المزخرف `@executor` لإنشاء منفذ مخصص يعرض النتيجة النهائية.

**المفاهيم الرئيسية:**
- `@executor(id="...")` - يسجل دالة كمنفذ في سير العمل
- `WorkflowContext[Never, str]` - تلميحات النوع للمدخلات/المخرجات
- `ctx.yield_output(...)` - ينتج النتيجة النهائية لسير العمل


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## الخطوة 5: تحميل متغيرات البيئة

قم بتكوين عميل LLM. يعمل هذا المثال مع:
- **نماذج GitHub** (المستوى المجاني مع رمز GitHub)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## الخطوة 6: إنشاء وكلاء ذكاء اصطناعي بمخرجات منظمة

نقوم بإنشاء **ثلاثة وكلاء متخصصين**، كل منهم ملفوف داخل `AgentExecutor`:

1. **availability_agent** - يتحقق من توفر الفنادق باستخدام الأداة
2. **alternative_agent** - يقترح مدنًا بديلة (عندما لا توجد غرف)
3. **booking_agent** - يشجع على الحجز (عندما تتوفر غرف)

**الميزات الرئيسية:**
- `tools=[hotel_booking]` - يزود الأداة للوكيل
- `response_format=PydanticModel` - يفرض مخرجات JSON منظمة
- `AgentExecutor(..., id="...")` - يلف الوكيل لاستخدامه في سير العمل


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## الخطوة 7: بناء سير العمل بالحواف الشرطية

الآن نستخدم `WorkflowBuilder` لبناء الرسم البياني مع التوجيه الشرطي:

**هيكل سير العمل:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**الطرق الرئيسية:**
- `.set_start_executor(...)` - يحدد نقطة الدخول
- `.add_edge(from, to, condition=...)` - يضيف حافة شرطية
- `.build()` - ينهي بناء سير العمل


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## الخطوة 8: تشغيل حالة الاختبار 1 - مدينة بدون توفر (باريس)

دعونا نختبر مسار **عدم التوفر** عن طريق طلب الفنادق في باريس (التي ليس لديها غرف في محاكاتنا).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## الخطوة 9: تشغيل حالة الاختبار 2 - المدينة مع التوفر (ستوكهولم)

الآن دعونا نختبر مسار **التوفر** عن طريق طلب الفنادق في ستوكهولم (التي تحتوي على غرف في محاكاةنا).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## النقاط الرئيسية والخطوات التالية

### ✅ ما تعلمته:

1. **نمط WorkflowBuilder**
   - استخدم `.set_start_executor()` لتحديد نقطة الدخول
   - استخدم `.add_edge(from, to, condition=...)` للتوجيه الشرطي
   - استدعِ `.build()` لإنهاء بناء سير العمل

2. **التوجيه الشرطي**
   - تقوم دوال الشروط بفحص `AgentExecutorResponse`
   - تحليل المخرجات المهيكلة لاتخاذ قرارات التوجيه
   - إرجاع `True` لتفعيل الحافة، و `False` لتخطيها

3. **تكامل الأدوات**
   - استخدم `@ai_function` لتحويل دوال بايثون إلى أدوات ذكاء اصطناعي
   - تقوم الوكلاء باستدعاء الأدوات تلقائيًا عند الحاجة
   - تعيد الأدوات JSON يمكن للوكلاء تحليله

4. **المخرجات المهيكلة**
   - استخدم نماذج Pydantic لاستخلاص بيانات آمنة من حيث النوع
   - اضبط `response_format=MyModel` عند إنشاء الوكلاء
   - حلل الردود باستخدام `Model.model_validate_json()`

5. **المُنفذون المخصصون**
   - استخدم `@executor(id="...")` لإنشاء مكونات سير العمل
   - يمكن للمُنفذين تحويل البيانات أو تنفيذ تأثيرات جانبية
   - استخدم `ctx.yield_output()` لإنتاج نتائج سير العمل

### 🚀 التطبيقات الواقعية:

- **حجوزات السفر**: التحقق من التوافر، اقتراح البدائل، مقارنة الخيارات
- **خدمة العملاء**: التوجيه بناءً على نوع المشكلة، المزاج، الأولوية
- **التجارة الإلكترونية**: التحقق من المخزون، اقتراح البدائل، معالجة الطلبات
- **مراجعة المحتوى**: التوجيه بناءً على درجات السمية، إشارات المستخدمين
- **سير عمل الموافقات**: التوجيه بناءً على المبلغ، دور المستخدم، مستوى المخاطرة
- **المعالجة متعددة المراحل**: التوجيه بناءً على جودة البيانات، الاكتمال

### 📚 الخطوات التالية:

- إضافة شروط أكثر تعقيدًا (معايير متعددة)
- تنفيذ الحلقات مع إدارة حالة سير العمل
- إضافة سير عمل فرعي للمكونات القابلة لإعادة الاستخدام
- التكامل مع واجهات برمجة التطبيقات الحقيقية (حجوزات الفنادق، أنظمة المخزون)
- إضافة معالجة للأخطاء ومسارات بديلة
- تصور سير العمل باستخدام أدوات التصور المدمجة


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
